# University Messy Dataset — Data Cleaning Practice Set

`master_dataset_raw.csv` — **1,005 rows × 15 columns**

This dataset is **deliberately messy**. Every quirk below was injected on
purpose so it maps to one exact step of the cleaning pipeline you outlined.
Nothing here is random noise you should ignore — each issue is something
to actively detect and fix.

---

## Column Reference

| Column                     | Type it *should* become | Messiness injected |
|---------------------------|--------------------------|---------------------|
| ` Student_ID `             | `string`                | Leading/trailing whitespace + mixed case in the **header itself**; ~3% of values are missing (unrecoverable → drop row) |
| `Full Name`                | `string`                | Clean (control column) |
| `Application Date`         | `datetime64`             | 5 different date string formats mixed together; ~3.5% missing |
| `Date of Birth`            | `datetime64`             | 3 different date string formats mixed together |
| `Age`                      | `Int64`                  | Impossible values injected (`-5`, `0`, `150`, `999`); ~4% missing |
| `Department`               | `category`/`string`      | Same real department spelled 4–5 different ways (typos, casing, double spaces) — e.g. `Entreprenuarship` / `Eentrepreneurship` / `entrepreneurship` |
| `Country`                  | `category`/`string`      | Clean (control column) |
| `Email`                    | `string`                 | Clean (control column) |
| `Fee_Paid`                 | `float`                  | Currency symbols mixed: `$4,843.15`, `Rs.3,810`, `2488.48 USD`, plain `2,136.02`; ~19% missing |
| `GPA`                      | `float`                  | A few values injected **above the 4.0 scale max** (5.0–9.9) — outliers; ~6% missing |
| `Attendance_Percentage`    | `float`                  | A few impossible percentages (`150.0`, `-20.0`, `105.5`); ~5% missing |
| `Is_Active`                | `bool`                   | Mixed representations: `True/False`, `TRUE/FALSE`, `1/0`, `Yes/No`, `yes/no`; ~11% missing |
| `Lab_Sensor_RPM`           | `float`                  | Simulated hardware-glitch outliers (`55000`, `48210`, `-100`, `99999`); ~3% missing |
| `Enrollment_Status`        | `category`/`string`      | ~30% missing/blank — a genuine categorical gap, ideal candidate for the `"Unknown"` flag technique |
| `Unnamed: 14`              | *(drop entirely)*        | 100% empty — a classic Excel "separator" column that should never survive Step 2 |

---

## What's baked in, mapped to your pipeline

**Step 1 — Profiling**
`df.shape` → `(1005, 15)`. `df.info()` will show every column as `object`
dtype even though several are really dates/numbers/booleans. `df.describe()`
on `Age`, `GPA`, `Attendance_Percentage`, and `Lab_Sensor_RPM` (after
forcing them numeric) will immediately flag the injected outliers.

**Step 2 — Structural standardization**
The ID column's header has stray spaces and mixed case on purpose
(` Student_ID `). `Unnamed: 14` is 100% null — a pure separator column
to drop.

**Step 3 — Deduplication**
30 rows are **exact** duplicates of existing rows. A further 25 rows
share a `Student_ID` with another row but have a different
`Application Date` / `Fee_Paid` — a realistic "resubmitted application"
partial-duplicate scenario, ideal for practicing `subset=` + `sort_values`
+ `keep='last'`.

**Step 4 — Type casting**
- `Application Date` / `Date of Birth`: 3–5 different string date formats mixed in the same column.
- `Fee_Paid`: currency symbols (`$`, `Rs.`), thousands separators, and trailing `"USD"` text.
- `Is_Active`: booleans represented as `True/False`, `1/0`, and `Yes/No` — all as strings.

**Step 5 — Missing values**
- `Student_ID` missing → these rows have no reliable key and should be **dropped**.
- `GPA` / `Attendance_Percentage` missing → good candidates for **mean/median imputation**.
- `Application Date` gaps → good candidate for **forward/backward fill** once sorted chronologically per student.
- `Enrollment_Status` missing (~30%) → flag as `"Unknown"` rather than dropping, since absence is itself meaningful here.

**Step 6 — Outliers & typos**
- `Age`: `-5`, `0`, `150`, `999` — impossible human ages.
- `GPA`: values above 4.0 on what should be a 4.0 scale.
- `Attendance_Percentage`: values above 100% or negative.
- `Lab_Sensor_RPM`: implausible sensor spikes simulating a hardware glitch.
- `Department`: the same real department deliberately spelled multiple ways — needs a mapping dictionary to consolidate into one canonical string per department.

**Step 7 — Final sanity check**
After you drop exact + partial duplicates and rows with missing
`Student_ID`, compare your final row count against the original 1,005 —
it should drop by roughly 30 (exact dupes) + up to 25 (partial dupes,
minus the ones you legitimately keep) + ~27 (missing IDs), landing
somewhere in the low 900s. Confirm zero unexpected nulls remain in any
column you chose to impute or flag, then export to `.csv` or `.parquet`.

---

## Suggested first commands to run

```python
import pandas as pd

df = pd.read_csv("master_dataset_raw.csv")

df.shape
df.info()
df.describe()
df.columns
```

From there, work through Steps 2–7 exactly as you outlined. Good luck —
this file should give you real reps on every technique in the pipeline.


## **Let's start Data Cleaning**(step-by-step)

**Step 1: Data Ingestion**

In [1]:
import pandas as pd
import numpy as np
import os

# File path.
file_path: str = '../messy_data_sets/university_messy_dataset.csv'

# Always check whether the file exists or not first.
if not os.path.exists(file_path):
    raise FileNotFoundError(f'Error! There is no such file {file_path}')
else:
    try:
        df: pd.DataFrame = pd.read_csv(
            file_path,
            # usecols = [list of columns] The memory Savior ingest only those columns which are necessary for the modeling ignore the rest.
            
            # dtype= {'column': datatype} This specifically use to say the pandas reader the data types of columns
            # NULL Standardization.
            na_values= [' ', '', 'Missing', 'missing', 'Unknown', 'unknown', 'N/A', 'NaN'],
            # Fault Tolerance
            on_bad_lines='warn', # If a row is corrupted (e.g., 6 columns instead of 5), skip it and log a warning instead of crashing.
            # PERFORMANCE
            engine= 'c' # Forces the underlying C-engine for maximum read speed.
        )
        print(f'Data Ingestion successful. Loaded {len(df)} rows into memory.')
    except Exception as e:
        print(f'PIPELINE FAILURE during CSV ingestion: {e}')


Data Ingestion successful. Loaded 1005 rows into memory.


**Step 2: Data Profiling and inspections**
- Check the structre of the data via using `df.shape()` method.
- Know about the dataset's pre assigned datatypes and non-null columns via `df.info()` mehtod.
- Get a quick statistical summary of numerical values in dataset via `df.info()` method.
- check the number of Null values in each columns via `df.isna().shape()`, however it may not be accurate, just because Null Values be actually Null. we humans read 'missing' 'unkown' 'NaN' as null values but in pandas it treats them as a valid and existing strings.

In [2]:
# number of rows and columns(sturcture of data).
dimensions: tuple = df.shape
print(f'The Dataset contains a total of {dimensions[0]} rows and {dimensions[1]} columns.')

The Dataset contains a total of 1005 rows and 15 columns.


In [3]:
# Let's check the dataset preassigned data types and non null values.
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1005 entries, 0 to 1004
Data columns (total 15 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0    Student_ID            978 non-null    str    
 1   Full Name              1005 non-null   str    
 2   Application Date       970 non-null    str    
 3   Date of Birth          1005 non-null   str    
 4   Age                    963 non-null    float64
 5   Department             1005 non-null   str    
 6   Country                1005 non-null   str    
 7   Email                  1005 non-null   str    
 8   Fee_Paid               817 non-null    str    
 9   GPA                    943 non-null    float64
 10  Attendance_Percentage  954 non-null    float64
 11  Is_Active              898 non-null    str    
 12  Lab_Sensor_RPM         978 non-null    float64
 13  Enrollment_Status      551 non-null    str    
 14  Unnamed: 14            0 non-null      float64
dtypes: float64(5), 

In [4]:
# Let's get a quick summary of numerical values inside the dataset.
df.describe()

,Age,GPA,Attendance_Percentage,Lab_Sensor_RPM,Unnamed: 14
count,963.000000,943.000000,954.000000,978.000000,0.0
mean,31.593977,2.986299,70.620231,2639.137832,NaN
std,11.642110,1.261650,20.530368,6688.492448,NaN
min,-5.000000,1.500000,-20.000000,-100.000000,NaN
25%,24.000000,2.130000,55.625000,1462.525000,NaN
50%,32.000000,2.890000,69.950000,2109.100000,NaN
75%,38.000000,3.510000,85.800000,2628.325000,NaN
max,150.000000,9.700000,150.000000,99999.000000,NaN


In [5]:
# let's get the total number of actual null-values(according to pandas) in every columns.
df.isna().sum()

# Insights

""" We'll work this in further steps:

1. The student Id columns has null values, which it shouldn't , So I gotta drop every rows which
   doesn't contains the student Id, Since there' no need of such rows.
2. We does has an empty column named 'Unnamed: 14' gotta drop whole column.
"""

" We'll work this in further steps:\n\n1. The student Id columns has null values, which it shouldn't , So I gotta drop every rows which\n   doesn't contains the student Id, Since there' no need of such rows.\n2. We does has an empty column named 'Unnamed: 14' gotta drop whole column.\n"

**Step 3: Structural Standardization**
- strip all the empty whitespaces around the column ID's.
- Make every columns looks standardized either uppercase or lower cases with Underscore e.g., `STUDENT_ID`, `student_id`.
- Drop all the empty columns, which are present for human readability , whcich would be named 'Unnamed: 14'

In [6]:
# To strip all the empty white spaces we're gonna use the .str method.
# and so for standardizing them to uppercases and replace the seperator empty
# space to underscore.

# sometimes we gotta force our dataframe columns datatype to be string manually
# it may happen that any of the column ID would be in different type and using
# the .str method on other datatypes rather then strings, the program would crash.

# df.columns = df.columns.astype(str) # wouldn't be necessary on this particular dataset.

df.columns = df.columns.str.strip().str.upper().str.replace(' ', '_')

# let's drop the empty columns presented just for human readability.
df: pd.DataFrame = df.dropna(axis=1, how='all')
df.columns.tolist() # Final columns of dataframe perfectly standardized, which won't lead to any crashes.


['STUDENT_ID',
 'FULL_NAME',
 'APPLICATION_DATE',
 'DATE_OF_BIRTH',
 'AGE',
 'DEPARTMENT',
 'COUNTRY',
 'EMAIL',
 'FEE_PAID',
 'GPA',
 'ATTENDANCE_PERCENTAGE',
 'IS_ACTIVE',
 'LAB_SENSOR_RPM',
 'ENROLLMENT_STATUS']

**Step 4: Deduplication**

Duplicates are categorised withing Two types.
- _Exact Duplicates_: All the elements of row A is exactly the same as of row B. It can happen by entring a data twice, or extracting the data from relation bases or Web scraping.
    - To handle those just drop the data by keeping either the first occurance or the last one.

- _Partial Duplicates_: inside multiple rows, the data from partial columsn(some of the columsn) contains the same values. e.g., if a student submitted an application form at 7 oct 2025, and again he/she submitted an application form at 15 Nov 2025, the student id's and almost data from both rows would be same however the submittion date would be different.
    - To handle such cases we must use logics to drop duplicates(like either we should keep the most recent date or the previous one).



In [7]:
# let's count duplicates values in each rows.
total_exact_duplicates: int = df.duplicated().sum()
print(f'Found total of {total_exact_duplicates} exact duplicate rows')

# View the actual duplicate rows (keep=False shows all copies of the duplicate)
# print("\n-------Problem rows--------\n")
#df[df.duplicated(keep=False)]

# let's drop the exact duplicates and assign this new dataframe to df_clean.
df_clean = df.drop_duplicates(ignore_index=True) # Ignore Index is very critical, to prevent miss indexing.

# now let's check how many exact duplicates my new dataframe df_clean have. expected 0.
total_dup_in_new_dataframe: int = df_clean.duplicated().sum()
print(f'Number of duplicates in new Dataframe {total_dup_in_new_dataframe}') 

Found total of 30 exact duplicate rows
Number of duplicates in new Dataframe 0


In [8]:
# Since we have zero exact duplicates.
# let's see how many duplicates does each columns contains.

spaces = 22 # Just for formating purpose only.
for columns in df_clean.columns:
    print(f'{columns} {' '* (spaces-len(columns))} {df_clean[columns].duplicated().sum()} duplicates.')

STUDENT_ID              50 duplicates.
FULL_NAME               297 duplicates.
APPLICATION_DATE        121 duplicates.
DATE_OF_BIRTH           37 duplicates.
AGE                     942 duplicates.
DEPARTMENT              944 duplicates.
COUNTRY                 965 duplicates.
EMAIL                   25 duplicates.
FEE_PAID                187 duplicates.
GPA                     697 duplicates.
ATTENDANCE_PERCENTAGE   508 duplicates.
IS_ACTIVE               964 duplicates.
LAB_SENSOR_RPM          75 duplicates.
ENROLLMENT_STATUS       970 duplicates.


**IMPORTANT NOTE: Before we move to the Partial Deduplication**

- We must understand **_Targeted Type Casting_**:

If we try to sort an `APPLICATION_DATE` column while it is still formatted as a raw string, Pandas will sort it alphabetically, not choronologically. "01-12-2025" Would mathematically appear before "10-01-2020".

This exposes a fundamental truth abotu data engineering that textbooks often gloss over.
**Pipelines are not always linear. They are driven by dependencies.**

**The Industry Solution: Targeted Casting**<br>
If a specific operation (like deduplication) relies on mathematical or chronological sorting, we must perform a targeted cast on that single dependency _before_ executing the structural drop. 


In [9]:
# let's type cast the APPLICATION_DATE and DATE_OF_BIRTH.

"""
Handling dates is notoriously one of the most frustrating tasks in data engineering.

Since, just Take a look at the APPLICATION_DATE Column we may encountered almost,
five different fomate of dates. Along with exact format strings used in Python's
strftime directive.

16/04/2022    - '%d/%m/%Y' 
08-Jul-2021   - '%d-%b-%Y' {%b: Abbreviated month name(e.g., Jul)}
2024-04-03    - '%Y-%m-%d'
11/25/2022    - '%m/%d/%Y'
May 08, 2021  - '%b %d, %Y' {%b: Abbreviated month name(e.g., Jul)}

Now we're gonna Use a Cascading Fallback pipline, As senior engineers do,
we explicitly define the exact formats that exist in our data, and use
combine_first() or fillna() to parse them one by one.
"""

# 1. Create an empty Series of NaT (Not a Time) with the length of our original dataset.
clean_dates_1 = pd.Series(pd.NaT, index=df_clean.index) # it'll will generate 975(actual len of dataset) NaT rows.
clean_dates_2 = pd.Series(pd.NaT, index=df_clean.index)
# NOTE: clean_dates_1 is for APPLICATION_DATE and clean_dates_2 is for DATE_OF_BIRTH

# 2. Define the formate we know exists in the raw data.
formats_to_try = [
    '%d/%m/%Y',  # 16/04/2022
    '%d-%b-%Y',  # 08-Jul-2021
    '%Y-%m-%d',  # 2024-04-03
    '%m/%d/%Y',  # 11/25/2022
    '%b %d, %Y'  # May 08, 2021
]

# Loop through each format and parse dates from df_clean['APPLICATION_DATE'],
# mathcing the format and replace it to our clean_dates Series, if the formate,
# doesn't match, it leaves to NaT.
# Then it tries format 2 ONLY on the remaining NaT rows.
for fmt in formats_to_try:
    # Attempt to parse
    parsed_attempt_1 = pd.to_datetime(df_clean['APPLICATION_DATE'], format=fmt, errors='coerce')
    parsed_attempt_2 = pd.to_datetime(df_clean['DATE_OF_BIRTH'], format=fmt, errors='coerce')

    # Fill in the blanks of our clean dates series with any succeful parses.
    clean_dates_1 = clean_dates_1.fillna(parsed_attempt_1)
    clean_dates_2 = clean_dates_2.fillna(parsed_attempt_2)

# At the last , Reassign the perfectly clean column back to the DataFrame.
df_clean['APPLICATION_DATE'] = clean_dates_1
df_clean['DATE_OF_BIRTH'] = clean_dates_2

df_clean.head()

,STUDENT_ID,FULL_NAME,APPLICATION_DATE,DATE_OF_BIRTH,AGE,DEPARTMENT,COUNTRY,EMAIL,FEE_PAID,GPA,ATTENDANCE_PERCENTAGE,IS_ACTIVE,LAB_SENSOR_RPM,ENROLLMENT_STATUS
0,STU00114,Farhan Farooqi,2022-04-16,1980-04-25,44.0,Data Sci.,India,farhan.farooqi114@gmail.com,"3,579.93",2.91,NaN,0,48210.0,NaN
1,STU00793,Riya Rana,2021-07-08,2002-12-13,21.0,Fiannce,India,riya.rana793@yahoo.com,NaN,NaN,41.4,yes,1430.7,Graduated
2,STU00741,Karan Kapoor,2021-06-06,1989-06-21,34.0,Electrical Engineering,Pakistan,karan.kapoor741@outlook.com,"$4,843.15",2.34,90.1,yes,1093.0,Enrolled
3,STU00432,Ava Siddiqui,2024-04-03,2004-03-31,20.0,Data Sci.,USA,ava.siddiqui432@yahoo.com,2488.48 USD,1.86,84.9,NaN,55000.0,Withdrawn
4,STU00243,Anjali Young,2024-03-16,1989-09-22,34.0,Business Administration,UAE,anjali.young243@outlook.com,"Rs.4,503",3.53,64.0,FALSE,2536.6,Withdrawn


**Let's move on to the partial Deduplication**

Since we have changed the Application_Date column to a certain datetime object, let's see which of the student have submitted there application form mistakenly twice at the same date, since we cannot directly drop every Duplicated `Student_ID`, it's because it isn't the primary key here, It can certainly happen when a student submit there application forms multiple times at different dates. So our primary key would become, the combined column of `STUDENT_ID` and `APPLICATION_DATE` which are called in terms **Composite keys**.

In [10]:
# Let's check do we have any Duplicates if we combine the STUDENT_ID and APPLICATION_DATE.
same_day_dupes = df_clean[df_clean.duplicated(subset=['STUDENT_ID', 'APPLICATION_DATE'], keep=False)]
same_day_dupes # Output zero which means we doesn't have any duplicates such like that.

,STUDENT_ID,FULL_NAME,APPLICATION_DATE,DATE_OF_BIRTH,AGE,DEPARTMENT,COUNTRY,EMAIL,FEE_PAID,GPA,ATTENDANCE_PERCENTAGE,IS_ACTIVE,LAB_SENSOR_RPM,ENROLLMENT_STATUS


**Step 5: Data Typecasting**<br>
Pandas generally import everything as generic type object (string) for play it safe. We have to force them to their original datatype.
- Conversion of the Datetime string into an original Datetime object.
- Remove the currency symbols `$` and give the value an original float data type.
- Handle the Binary interpretations like `1` or `0` , `True` or `False` , `one` or `zero` to a standard boolean type.


In [11]:
# Since we have already converted our datetime string into a pandas datetime object, let's move
# To the next one. "Strinping out the currency symbol".

"""
Into an enterprise the financial data will not be found cleanly, always mixed with sum currency
symbol $ or with commas , or some strings like "Contact Admin", and if we try to get the some
of the values, the pandas try to concatanate values ending memory crash or garbage outputs.
"""

# The Amatuers Approach.

# In this Approach we choose to hardcode every possible Alphabets or symbols are present in the 
# Formate, and It would be quite a difficult task for large data sets, containing multiple formats.
# df_clean['FEE_PAID'] = df_clean['FEE_PAID'].str.replace('$', "")
# df_clean['FEE_PAID'] = df_clean['FEE_PAID'].str.replace(',', "")

# # And even if we think that we hard coded every possible Alphabets and symbols, if we miss any 
# # String which can not be converted into numeric values , it will throw an error.
# df_clean['FEE_PAID'] = df_clean['FEE_PAID'].astype(float) # Output : ValueError

# The industry standard Approach is using the regex (Regular Expressions), instead of saying what
# to remove in pandas, we will say what to keep(e.g. keep every numercial values with the decimals.)


# 1. force our columns data type into string , to prevent error caused by existing NaNs.
df_clean['FEE_PAID'] = df_clean['FEE_PAID'].astype(str)

# The REGEX shield: replace everything that is not a digits or decimal.
df_clean['FEE_PAID'] = df_clean['FEE_PAID'].str.replace(r'[^\d.]','', regex=True)

# Force them to the numeric type and fill the empty spaces to NaNs.
df_clean['FEE_PAID'] = pd.to_numeric(df_clean['FEE_PAID'], errors='coerce',)

print(f'The "FEE_PAID" column data type is now {df_clean['FEE_PAID'].dtype}')



# NOTE: The Theoratical Breakdown:

# r'[^\d.]' is the heart of this system. the '^' inside the braket tells 'Not too', '\d' means
# any digits '.' means any periods. Basically it says replace those which are not digits.

# 'regex=True' is telling pandas to use the C-based regex engine to perform the operation in a 
# lightning fast speed to to the millions of rows.

# 'pd.to_numeric' along with "errors='coerce'" tell pandas to convert the rows into numeric values
# and handle the empty cells with NaNs.


The "FEE_PAID" column data type is now float64


In [12]:
# let's move on to the next one which handling the Binary terms with a regular boolean values.

# The Cause for the direct approach.

# df_clean['IS_ACTIVE'] = df_clean['IS_ACTIVE'].astype(bool) 
# What's wrong with this approach: The pandas turn every cell containing any values to 'TRUE'
# no matter if it would be 'True' or 'False' or '1' or '0' every cells becomes TRUE and Corrupt
# our entire dataset.

# The Industry approach: The Mapping Dcitionaries.

# 1. Define the absolute truth parameters [the dictionary].
bool_mapping: dict[str, bool] = {
    'true': True, 't': True, '1': True, 'yes': True,
    'false': False, 'f': False, '0': False, 'no': False
}

# 2. Forcefully convert the columns into string and strip every white spaces  and convert them
# into lower cases.

# We cast it to a raw strings first incase there are actual numbers of mapping is.
raw_strings = df_clean['IS_ACTIVE'].astype(str).str.strip().str.lower()

# Apply the mapping function with the bool_mapping parameters.
df_clean['IS_ACTIVE'] = raw_strings.map(bool_mapping)

# Finally just cast datatype into bool forcefully.
df_clean['IS_ACTIVE'] = df_clean['IS_ACTIVE'].astype('boolean')

df_clean["IS_ACTIVE"].dtype

BooleanDtype

**Now we covered two milestones, let's move on to some regular typcastings**<br>


In [21]:
# panda assigned Age column a float datatype , however it should be an integer.
# df_clean['AGE'] = df_clean['AGE'].astype('int64') 
# By running this code block will produce an IntCastingNaNError, which generally
# Says pandas cannot cast a null or NaN values as an integer, to hold such 
# specific error pandas introduced a new data type 'Int64' with capital 'I'
# instead of using 'int' or 'int64' we will use the latest 'Int64' datatype.

df_clean['AGE'] = df_clean['AGE'].astype('Int64')
# and Voila` we've conveted whole column into integer along with NaN values.

df_clean.head()

,STUDENT_ID,FULL_NAME,APPLICATION_DATE,DATE_OF_BIRTH,AGE,DEPARTMENT,COUNTRY,EMAIL,FEE_PAID,GPA,ATTENDANCE_PERCENTAGE,IS_ACTIVE,LAB_SENSOR_RPM,ENROLLMENT_STATUS
0,STU00114,Farhan Farooqi,2022-04-16,1980-04-25,44,Data Sci.,India,farhan.farooqi114@gmail.com,3579.9300,2.91,NaN,False,48210.0,NaN
1,STU00793,Riya Rana,2021-07-08,2002-12-13,21,Fiannce,India,riya.rana793@yahoo.com,NaN,NaN,41.4,True,1430.7,Graduated
2,STU00741,Karan Kapoor,2021-06-06,1989-06-21,34,Electrical Engineering,Pakistan,karan.kapoor741@outlook.com,4843.1500,2.34,90.1,True,1093.0,Enrolled
3,STU00432,Ava Siddiqui,2024-04-03,2004-03-31,20,Data Sci.,USA,ava.siddiqui432@yahoo.com,2488.4800,1.86,84.9,<NA>,55000.0,Withdrawn
4,STU00243,Anjali Young,2024-03-16,1989-09-22,34,Business Administration,UAE,anjali.young243@outlook.com,0.4503,3.53,64.0,False,2536.6,Withdrawn


**Some minor but , effective changes to fully my dataset for industry level purpose**<br>

For now, we have successfully implemented the hardest parts of data type casting. Our `datetime64` conversions are locked in , our booleans are using the nullable extension array, and we have successfully deployed `Int64` to prevent the NaN crashes.

From a purely functional standpoint, the pipeline will run. 

However, from an **industry-level performance and memory optimization** standpoint, we are missing one final, critical casting step.

We have left all our text columns as standard strings (`str`). We need to apply **Categorical casting** and **Nullable String Casting**.

1. **The Memory Drain: Categorical Casting**
Look at `DEPARTMENt`, `COUNTRY`, and `ENROLLMENT_STATUS`.
These are low-cardinality columns. This means out of you 975 rows there are probably only 5 distinct departments, maybe 20 countries, and 3 enrollment statuses (e.g, Enrolled, Graduated, Dropped, etc).

Right Now, Pandas is saving the string `"Computer Science"` in memory hundreds of times. if this dataset scales to 5 million rows, our RAM will max out instantly.

- **The Fix**: Cast low-cardinality text columns to `category`.
- **The Theory**: Pandas will store the string `"Computer Science"` in memory exactly one, assign it an integer code (like `0`), and just use the integer undert the hood. This can reduce memory usage by up to 90 % and makes `.groupby()` operations lightning fast.

2. **The String `NaN` Trap: Nullable String Casting**<br>
Look at our `STUDENT_ID`, `FULL_NAME`, and `EMAIL`.
These are high-cardinality columns (mostly unique), so they should remain strings. However, we casted them using Python's native `.astype(str)`, we likely triggered a massive data corruption erro.

When Python forces a `NaN` value into a native `str`. it doesn't leave it as a missing vlaue. it leterally types out the letters `'nan'`. When we move to Step 5 to handle missing values, `.isna()` will return `False` because pandas thinks a student's name is literally 'nan'.
- **The Fix**: we must use the Pandas nullable string extension type, exactly like we did wit `Int64` and `boolean`. we use `'string'` (which calls the `StringDtype` under the hood) istead of the python `str` function.

In [28]:
# Number 1 Fix: converting low-cardinality texts into category rather than strings.
cols_to_category: list = ['DEPARTMENT', 'COUNTRY', 'ENROLLMENT_STATUS']

for cols in cols_to_category:
    df_clean[cols] = df_clean[cols].astype('category')

# Number 2 Fix: converting high-cardinality texts into 'string' datatype for 
# preventing from NaN values crashes.
cols_to_string: list = ['STUDENT_ID', 'FULL_NAME', 'EMAIL']

for cols in cols_to_string:
    df_clean[cols] = df_clean[cols].astype('string')

df_clean.info()
df_clean.head()

<class 'pandas.DataFrame'>
RangeIndex: 975 entries, 0 to 974
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   STUDENT_ID             949 non-null    string        
 1   FULL_NAME              975 non-null    string        
 2   APPLICATION_DATE       940 non-null    datetime64[ns]
 3   DATE_OF_BIRTH          975 non-null    datetime64[ns]
 4   AGE                    934 non-null    Int64         
 5   DEPARTMENT             975 non-null    category      
 6   COUNTRY                975 non-null    category      
 7   EMAIL                  975 non-null    string        
 8   FEE_PAID               790 non-null    float64       
 9   GPA                    914 non-null    float64       
 10  ATTENDANCE_PERCENTAGE  925 non-null    float64       
 11  IS_ACTIVE              875 non-null    boolean       
 12  LAB_SENSOR_RPM         948 non-null    float64       
 13  ENROLLMENT_STATU

,STUDENT_ID,FULL_NAME,APPLICATION_DATE,DATE_OF_BIRTH,AGE,DEPARTMENT,COUNTRY,EMAIL,FEE_PAID,GPA,ATTENDANCE_PERCENTAGE,IS_ACTIVE,LAB_SENSOR_RPM,ENROLLMENT_STATUS
0,STU00114,Farhan Farooqi,2022-04-16,1980-04-25,44,data sci.,india,farhan.farooqi114@gmail.com,3579.9300,2.91,NaN,False,48210.0,NaN
1,STU00793,Riya Rana,2021-07-08,2002-12-13,21,fiannce,india,riya.rana793@yahoo.com,NaN,NaN,41.4,True,1430.7,graduated
2,STU00741,Karan Kapoor,2021-06-06,1989-06-21,34,electrical engineering,pakistan,karan.kapoor741@outlook.com,4843.1500,2.34,90.1,True,1093.0,enrolled
3,STU00432,Ava Siddiqui,2024-04-03,2004-03-31,20,data sci.,usa,ava.siddiqui432@yahoo.com,2488.4800,1.86,84.9,<NA>,55000.0,withdrawn
4,STU00243,Anjali Young,2024-03-16,1989-09-22,34,business administration,uae,anjali.young243@outlook.com,0.4503,3.53,64.0,False,2536.6,withdrawn


In [35]:
colums_list: list = df_clean.columns.to_list()

spaces = 25
for column in colums_list:
    print(f"{column}{" "*(spaces-len(column))} {df_clean[column].isna().sum()}")

df_clean[df_clean['STUDENT_ID'].isna()]

STUDENT_ID                26
FULL_NAME                 0
APPLICATION_DATE          35
DATE_OF_BIRTH             0
AGE                       41
DEPARTMENT                0
COUNTRY                   0
EMAIL                     0
FEE_PAID                  185
GPA                       61
ATTENDANCE_PERCENTAGE     50
IS_ACTIVE                 100
LAB_SENSOR_RPM            27
ENROLLMENT_STATUS         439


,STUDENT_ID,FULL_NAME,APPLICATION_DATE,DATE_OF_BIRTH,AGE,DEPARTMENT,COUNTRY,EMAIL,FEE_PAID,GPA,ATTENDANCE_PERCENTAGE,IS_ACTIVE,LAB_SENSOR_RPM,ENROLLMENT_STATUS
23,<NA>,Neha Taylor,2021-11-09,1997-05-17,27,entrepreneurship,uk,neha.taylor653@gmail.com,0.2261,2.76,77.6,False,1853.3,pending
52,<NA>,Noah Bose,2023-09-03,2002-08-03,21,entrepreneurship,germany,noah.bose619@outlook.com,4806.0700,3.37,51.8,False,2733.1,pending
65,<NA>,Meera Sharma,2023-10-14,1985-04-02,39,finance,usa,meera.sharma141@outlook.com,1071.1900,1.80,60.1,True,953.3,NaN
76,<NA>,Noah Young,2021-09-25,2004-08-18,19,finance,uk,noah.young488@outlook.com,0.1372,3.52,55.6,True,3043.9,withdrawn
197,<NA>,Vivaan Joshi,2024-11-28,1983-01-15,41,computer science,uk,vivaan.joshi683@yahoo.com,3122.3300,1.70,70.5,True,1165.4,pending
231,<NA>,Sara Malik,2021-05-20,1990-06-05,34,bus. admin,canada,sara.malik286@yahoo.com,3763.6300,1.56,96.3,<NA>,1320.4,enrolled
284,<NA>,Farhan Chaudhary,2022-02-17,1989-07-19,34,computer science,germany,farhan.chaudhary265@outlook.com,3951.6500,2.35,77.0,True,1265.6,withdrawn
433,<NA>,Priya Walker,2022-11-12,2002-03-09,22,electrical engineering,uae,priya.walker18@yahoo.com,4043.1900,1.60,52.0,False,2769.3,NaN
454,<NA>,Omar Nair,2021-04-24,1992-01-06,32,finance,bangladesh,omar.nair527@gmail.com,2986.2300,2.44,82.1,True,1851.1,pending
476,<NA>,Farhan Mehta,2021-06-06,1999-10-31,24,comp. science,india,farhan.mehta757@yahoo.com,NaN,1.68,98.6,False,1329.8,pending
